In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["DEBUG"] = "0"
os.environ["HF_HOME"] = "/scratch/hf_cache"


In [2]:
import torch
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
from vllm import LLM, SamplingParams

INFO 11-26 18:08:04 [__init__.py:216] Automatically detected platform cuda.


In [4]:
# llm = LLM(model="LiquidAI/LFM2-2.6B",
#           dtype="bfloat16",
#           gpu_memory_utilization=0.9,
#           tensor_parallel_size=1,
#           enable_expert_parallel=False,
#           enforce_eager=False)

In [5]:
# llm = LLM(model="ibm-granite/granite-4.0-tiny-preview",
#           dtype="bfloat16",
#           gpu_memory_utilization=0.4,
#           tensor_parallel_size=1,
#           enable_expert_parallel=False,
#           enforce_eager=False)

In [6]:
# llm = LLM(model="nvidia/Nemotron-H-8B-Reasoning-128K",
#           dtype="bfloat16",
#           gpu_memory_utilization=0.4,
#           tensor_parallel_size=1,
#           enable_expert_parallel=False,
#           enforce_eager=True,
#           trust_remote_code=True)

In [7]:
# llm = LLM(model="ibm-ai-platform/Bamba-9B-v2",
#           dtype="bfloat16",
#           max_model_len=4096,
#           gpu_memory_utilization=0.4,
#           tensor_parallel_size=1,
#           enable_expert_parallel=False,
#           enforce_eager=True)

In [ ]:
llm = LLM(model="/scratch/checkpoints/smoe/hf/midtrain_phase2_decay/iter_0030036",
          dtype="bfloat16",
          gpu_memory_utilization=0.8,
          max_num_batched_tokens=20,
          block_size=8,
          tensor_parallel_size=1,
          data_parallel_size=1,
          enable_expert_parallel=True,
          enforce_eager=False)

INFO 11-26 18:08:09 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'enable_expert_parallel': True, 'block_size': 8, 'gpu_memory_utilization': 0.8, 'max_num_batched_tokens': 20, 'disable_log_stats': True, 'model': '/scratch/checkpoints/smoe/hf/midtrain_phase2_decay/iter_0030036'}
INFO 11-26 18:08:09 [model.py:547] Resolved architecture: SMoEForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 11-26 18:08:09 [model.py:1526] Using max model len 32768
INFO 11-26 18:08:09 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=20.
INFO 11-26 18:08:09 [config.py:297] Hybrid or mamba-based model detected: disabling prefix caching since it is not yet supported.
INFO 11-26 18:08:09 [config.py:308] Hybrid or mamba-based model detected: setting cudagraph mode to FULL_AND_PIECEWISE in order to optimize performance.
INFO 11-26 18:08:11 [config.py:376] Setting attention block size to 16 tokens to ensure that attention page size is >= mamba page size.
INFO 11-26 18:08:11 [config.py:397] Padding mamba page size by 77.78% to ensure that mamba page size and attention page size are exactly equal.
(EngineCore_DP0 pid=2348700) INFO 11-26 18:08:12 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=2348700) INFO 11-26 18:08:12 [core.py:77] Initializing a V1 LLM engine (v0.10.0rc2.dev2134+g20373a0a2) with config: model='/scratch/checkpoints/smoe/hf/m

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 2.48k/2.48k [00:03<00:00, 740weights/s]


(EngineCore_DP0 pid=2348700) INFO 11-26 18:08:18 [default_loader.py:267] Loading weights took 3.95 seconds
(EngineCore_DP0 pid=2348700) INFO 11-26 18:08:18 [gpu_model_runner.py:2653] Model loading took 16.4708 GiB and 4.270730 seconds
(EngineCore_DP0 pid=2348700) INFO 11-26 18:08:26 [backends.py:548] Using cache directory: /data/home/yury/.cache/vllm/torch_compile_cache/f2cf69fd66/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2348700) INFO 11-26 18:08:26 [backends.py:559] Dynamo bytecode transform time: 7.46 s
(EngineCore_DP0 pid=2348700) INFO 11-26 18:08:28 [backends.py:197] Cache the graph for dynamic shape for later use
(EngineCore_DP0 pid=2348700) INFO 11-26 18:08:53 [backends.py:218] Compiling a graph for dynamic shape takes 26.10 s
(EngineCore_DP0 pid=2348700) INFO 11-26 18:08:54 [fused_moe.py:788] Using configuration from /data/home/yury/workspace/smoe/Zvllm-v1/vllm/model_executor/layers/fused_moe/configs/E=16,N=2048,device_name=NVIDIA_H100_80GB_HBM3.json for Mo

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 5/5 [00:03<00:00,  1.65it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 5/5 [00:00<00:00,  5.58it/s]


(EngineCore_DP0 pid=2348700) INFO 11-26 18:09:06 [gpu_model_runner.py:3480] Graph capturing finished in 5 secs, took 0.39 GiB
(EngineCore_DP0 pid=2348700) INFO 11-26 18:09:06 [core.py:210] init engine (profile, create kv cache, warmup model) took 47.22 seconds
INFO 11-26 18:09:07 [llm.py:306] Supported_tasks: ['generate']


In [9]:
prompts = [
    # "Question: What factors contributed to the fall of the Roman Empire? Answer: ",
    # "Question: The president of the United States is? Answer: ",
    "Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ",
    # "Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ",
    # "Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ",
    # "Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ",
]
sampling_params = SamplingParams(max_tokens=100, temperature=0)
outputs = llm.generate(prompts, sampling_params)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"\nPrompt: {prompt!r},\nGenerated text: {generated_text!r}")

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Prompt: 'Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ',
Generated text: '12 Explanation:\nIn one season a flower blooms three times. In one year, there is one blooming season.\nIn two years, there are two blooming seasons.\nIn one season, a flower blooms three times.\nIn two years, a flower blooms six times.\nIn two years, two flowers bloom twelve times.\nIn two years, two flowers bloom twelve times.\nIn two years, two flowers bloom twelve times.\nIn two years, two flowers bloom twelve times'


In [10]:
prompts = [
    "Question: What factors contributed to the fall of the Roman Empire? Answer: ",
]
sampling_params = SamplingParams(max_tokens=100, temperature=0)
outputs = llm.generate(prompts, sampling_params)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"\nPrompt: {prompt!r},\nGenerated text: {generated_text!r}")

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Prompt: 'Question: What factors contributed to the fall of the Roman Empire? Answer: ',
Generated text: '1. Invasions by Barbarian tribes 2. Economic troubles and overreliance on slave labor 3. The rise of the Eastern Empire 4. Overexpansion and military overspending 5. Government corruption and political instability 6. The arrival of the Huns and the migration of the Barbarians 7. Christianity and the loss of traditional values 8. Weakening of the Roman legions\nQuestion: What was the significance of the Magna Carta? Answer: The Magna Carta was a'


In [11]:
prompts = [
    "Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ",
    "Question: What factors contributed to the fall of the Roman Empire? Answer: ",
    # "Question: The president of the United States is? Answer: ",
]
sampling_params = SamplingParams(max_tokens=100, temperature=0)
outputs = llm.generate(prompts, sampling_params)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"\nPrompt: {prompt!r},\nGenerated text: {generated_text!r}")

Adding requests:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Prompt: 'Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ',
Generated text: '12 Explanation:\nIn one season a flower blooms three times. In one year, there is one blooming season.\nIn two years, there are two blooming seasons.\nIn one season, a flower blooms three times.\nIn two years, a flower blooms six times.\nIn two years, two flowers bloom twelve times.\nIn two years, two flowers bloom twelve times.\nIn two years, two flowers bloom twelve times.\nIn two years, two flowers bloom twelve times'

Prompt: 'Question: What factors contributed to the fall of the Roman Empire? Answer: ',
Generated text: '1. Invasions by Barbarian tribes 2. Economic troubles and overreliance on slave labor 3. The rise of the Eastern Empire 4. Overexpansion and military overspending 5. Government corruption and political instability 6. The arrival of the Huns and the migration

In [12]:
prompts = [
    "Question: What factors contributed to the fall of the Roman Empire? Answer: ",
    "Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ",
    # "Question: The president of the United States is? Answer: ",
]
sampling_params = SamplingParams(max_tokens=100, temperature=0)
outputs = llm.generate(prompts, sampling_params)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"\nPrompt: {prompt!r},\nGenerated text: {generated_text!r}")

Adding requests:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Prompt: 'Question: What factors contributed to the fall of the Roman Empire? Answer: ',
Generated text: '1. Invasions by Barbarian tribes 2. Economic troubles and overreliance on slave labor 3. The rise of the Eastern Empire 4. Overexpansion and military overspending 5. Government corruption and political instability 6. The arrival of the Huns and the migration of the Barbarians 7. Christianity and the loss of traditional values 8. Weakening of the Roman legions\nQuestion: What was the significance of the Magna Carta? Answer: The Magna Carta was a'

Prompt: 'Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ',
Generated text: '12 Explanation:\nIn one season a flower blooms three times. In one year, there is one blooming season.\nIn two years, there are two blooming seasons.\nIn one season, a flower blooms three times.\nIn two years, a flower blooms six tim

In [13]:
prompts = [
    "Question: What factors contributed to the fall of the Roman Empire? Answer: ",
    "Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ",
    "Question: The president of the United States is? Answer: ",
    "Question: What factors contributed to the fall of the Roman Empire? Answer: ",
    "Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ",
    "Question: The president of the United States is? Answer: ",
]
sampling_params = SamplingParams(max_tokens=100, temperature=0)
outputs = llm.generate(prompts, sampling_params)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"\nPrompt: {prompt!r},\nGenerated text: {generated_text!r}")

Adding requests:   0%|          | 0/6 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/6 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Prompt: 'Question: What factors contributed to the fall of the Roman Empire? Answer: ',
Generated text: '1. Invasions by Barbarian tribes 2. Economic troubles and overreliance on slave labor 3. The rise of the Eastern Empire 4. Overexpansion and military overspending 5. Government corruption and political instability 6. The arrival of the Huns and the migration of the Barbarians 7. Christianity and the loss of traditional values 8. Weakening of the Roman legions\nQuestion: What was the significance of the Magna Carta? Answer: The Magna Carta was a'

Prompt: 'Question: In one season a flower blooms three times. In one year, there is one blooming season. How many times do two flowers bloom in two years? Please include your logic. Answer: ',
Generated text: '12 Explanation:\nIn one season a flower blooms three times. In one year, there is one blooming season.\nIn two years, there are two blooming seasons.\nIn one season, a flower blooms three times.\nIn two years, a flower blooms six tim